# RRT Internals: Building the Tree One Step at a Time

## What does RRT actually do?

The **Rapidly-exploring Random Tree** (RRT) is the workhorse of sampling-based motion planning. Libraries like OMPL have polished, multi-threaded implementations — but the core idea is shockingly simple. Here it is in 4 steps:

1. **Sample**: Pick a random configuration `q_rand` anywhere in C-space
2. **Nearest**: Find the node in the current tree closest to `q_rand`
3. **Steer**: Move from that nearest node toward `q_rand` by at most `step_size`
4. **Extend**: If the new configuration is collision-free, add it to the tree

Repeat until the tree reaches the goal (or a timeout).

That's it. No gradient descent, no potential fields, no explicit obstacle map. Just: sample, step, check, add.

### Why does it work?

RRT has a beautiful property called the **Voronoi bias**: nodes in large unexplored regions of C-space are more likely to be the *nearest* to a random sample, so the tree naturally expands toward unexplored space. This is not an explicit heuristic — it emerges organically from the nearest-neighbor step.

This notebook implements a minimal 2D RRT from scratch to make every step visible.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import defaultdict

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

In [ ]:
# 2D planning problem
# C-space = [-pi, pi] x [-pi, pi] (same as for a 2-DOF arm from notebook 01)
C_MIN = -np.pi
C_MAX = np.pi

# Start and goal
q_start = np.array([-2.5, -2.5])
q_goal  = np.array([ 2.5,  2.5])
GOAL_RADIUS = 0.3   # how close we need to get to declare success

# Rectangular forbidden zone in C-space
# Format: [(x_min, x_max, y_min, y_max), ...]
OBSTACLES = [
    (-1.5,  0.5, -0.8,  1.5),   # large diagonal blocker
    ( 0.3,  2.0, -2.0, -0.3),   # lower-right obstacle
]

def point_in_obstacle(q):
    """Return True if q falls inside any rectangular obstacle."""
    for (xmin, xmax, ymin, ymax) in OBSTACLES:
        if xmin <= q[0] <= xmax and ymin <= q[1] <= ymax:
            return True
    return False

print(f"Start: {q_start}")
print(f"Goal:  {q_goal}")
print(f"Goal radius: {GOAL_RADIUS}")
print(f"Start in collision: {point_in_obstacle(q_start)}")
print(f"Goal  in collision: {point_in_obstacle(q_goal)}")

In [ ]:
# ============================================================
# Core RRT components — each function is one logical step
# ============================================================

def sample_random(rng):
    """Uniform sample anywhere in C-space."""
    return rng.uniform(C_MIN, C_MAX, size=2)

def nearest(tree_nodes, q):
    """Return the node in the tree closest to q (Euclidean)."""
    tree_arr = np.array(tree_nodes)
    dists = np.linalg.norm(tree_arr - q, axis=1)
    return tree_nodes[np.argmin(dists)]

def steer(q_near, q_rand, step=0.3):
    """Move from q_near toward q_rand by at most `step` distance."""
    diff = q_rand - q_near
    dist = np.linalg.norm(diff)
    if dist <= step:
        return q_rand.copy()  # q_rand is already close enough — go directly
    return q_near + (diff / dist) * step

def is_collision_free(q):
    """True if q is within bounds and not inside any obstacle."""
    if not (C_MIN <= q[0] <= C_MAX and C_MIN <= q[1] <= C_MAX):
        return False
    return not point_in_obstacle(q)

def connect(tree_nodes, parent, q_rand, step=0.3):
    """
    Full RRT extend step:
      1. Find nearest node
      2. Steer toward q_rand
      3. Add new node if collision-free
    Returns (q_new, q_near) if successful, else (None, None).
    """
    q_near = nearest(tree_nodes, q_rand)
    q_new = steer(q_near, q_rand, step)
    if is_collision_free(q_new):
        tree_nodes.append(q_new)
        parent[tuple(q_new)] = tuple(q_near)
        return q_new, q_near
    return None, None

print("RRT components defined.")
print()
print("Quick sanity check:")
q_a = np.array([0.0, 0.0])
q_b = np.array([1.0, 1.0])
q_steered = steer(q_a, q_b, step=0.3)
print(f"  steer([0,0], [1,1], 0.3) = {np.round(q_steered, 4)}  (should be ~0.212 apart from start)")
print(f"  distance = {np.linalg.norm(q_steered - q_a):.4f}  (should be 0.3)")

In [ ]:
def run_rrt(seed=42, max_iter=500, step_size=0.3):
    """Run RRT and return (tree_nodes, parent_map, path, found)."""
    rng = np.random.default_rng(seed)
    
    tree_nodes = [q_start.copy()]
    parent = {}  # child -> parent (keyed by tuple for hashability)
    parent[tuple(q_start)] = None
    
    found = False
    goal_node = None
    
    for i in range(max_iter):
        # 5% of the time, sample the goal directly (goal biasing)
        if rng.random() < 0.05:
            q_rand = q_goal.copy()
        else:
            q_rand = sample_random(rng)
        
        q_new, _ = connect(tree_nodes, parent, q_rand, step=step_size)
        
        if q_new is not None:
            if np.linalg.norm(q_new - q_goal) < GOAL_RADIUS:
                found = True
                goal_node = q_new
                break
    
    # Reconstruct path by walking parent pointers
    path = []
    if found:
        node = tuple(goal_node)
        while node is not None:
            path.append(np.array(node))
            node = parent[node]
        path.reverse()
    
    return tree_nodes, parent, path, found

# Run RRT with seed 42
tree_nodes, parent, path, found = run_rrt(seed=42)
print(f"RRT result (seed=42):")
print(f"  Tree size: {len(tree_nodes)} nodes")
print(f"  Found path: {found}")
if found:
    print(f"  Path length (nodes): {len(path)}")
    path_len = sum(np.linalg.norm(path[i+1] - path[i]) for i in range(len(path)-1))
    print(f"  Path length (C-space distance): {path_len:.3f}")

In [ ]:
def draw_rrt(tree_nodes, parent, path, ax, title):
    """Draw the RRT tree, obstacles, path, start, and goal."""
    
    # Draw obstacles
    for (xmin, xmax, ymin, ymax) in OBSTACLES:
        rect = patches.Rectangle(
            (xmin, ymin), xmax - xmin, ymax - ymin,
            linewidth=1.5, edgecolor='darkred', facecolor='red', alpha=0.35
        )
        ax.add_patch(rect)
    
    # Draw tree edges (thin gray)
    for node, par in parent.items():
        if par is not None:
            ax.plot(
                [node[0], par[0]], [node[1], par[1]],
                color='lightgray', linewidth=0.6, alpha=0.7, zorder=1
            )
    
    # Draw path (thick blue)
    if path:
        px = [p[0] for p in path]
        py = [p[1] for p in path]
        ax.plot(px, py, color='royalblue', linewidth=2.5, zorder=3, label='Path')
    
    # Start and goal
    ax.scatter(*q_start, color='limegreen', s=120, zorder=5, label='Start')
    ax.scatter(*q_goal, color='red', s=120, marker='*', zorder=5, label='Goal')
    goal_circle = plt.Circle(q_goal, GOAL_RADIUS, fill=False, color='red', linestyle=':', linewidth=1.5)
    ax.add_patch(goal_circle)
    
    ax.set_xlim(C_MIN - 0.1, C_MAX + 0.1)
    ax.set_ylim(C_MIN - 0.1, C_MAX + 0.1)
    ax.set_xlabel('θ1 [rad]')
    ax.set_ylabel('θ2 [rad]')
    ax.set_title(title, fontsize=11)
    ax.set_aspect('equal')
    ax.legend(loc='upper left', fontsize=8)
    ax.text(0.02, 0.02, f"{len(tree_nodes)} nodes", transform=ax.transAxes,
            fontsize=9, color='gray')

fig, ax = plt.subplots(figsize=(7, 7))
draw_rrt(tree_nodes, parent, path, ax,
         'RRT tree — seed=42\n(gray=tree edges, blue=found path, red=forbidden)')
plt.tight_layout()
plt.show()

## The Voronoi Bias — Exploration Without Explicit Heuristics

Look at how the tree explores the space. It does **not** fill uniformly — it pushes outward into unexplored regions.

This happens because of a geometric property:
- When you sample a random point `q_rand` in a region with few tree nodes, the *nearest* tree node is likely far away
- The tree extends toward that region, reducing its "Voronoi cell" (the region where it's the nearest node)
- Regions that already have many nodes are less likely to attract a new sample (their Voronoi cells are small)

The tree **self-regulates** its exploration. No explicit "frontier tracking" needed.

Now let's see how the tree shape changes with a different random seed — this is **probabilistic completeness** in action.

In [ ]:
seeds = [7, 99, 314, 1337]
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

for ax, seed in zip(axes.ravel(), seeds):
    tn, par, pth, fnd = run_rrt(seed=seed)
    status = f"Found ({len(pth)} nodes)" if fnd else "No path found"
    draw_rrt(tn, par, pth, ax, f'seed={seed} — {status}')

plt.suptitle('RRT with different random seeds — same problem, different tree shapes',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Notice:")
print("- Each seed produces a completely different tree topology")
print("- But they all find (or fail to find) the same path through the same obstacles")
print("- Some seeds find shorter paths, some longer — RRT is NOT optimal")
print("- RRT* would rewire the tree to progressively improve path quality")

## Exercises

**1. Effect of step size**  
The `step_size` controls how far each extend step reaches. Try:

```python
tn, par, pth, fnd = run_rrt(seed=42, step_size=0.05)
```

- What happens to the tree? More nodes? More detail?
- Does it still find a path in 500 iterations?
- Now try `step_size=1.5`. What goes wrong?

**2. Goal biasing**  
In the `run_rrt` function above, 5% of samples are the goal directly. This is **goal biasing** — it speeds up convergence near the goal.
- Change it to 0% (pure random) and 50% (heavy bias). How does the tree look?
- Heavy goal bias makes the tree look like a straight shot from start to goal — why is that problematic with obstacles?

**3. RRT vs grid search**  
A grid-based planner (like A*) would discretize the C-space into a grid and search it. For our 2D problem a 100×100 grid has 10,000 cells. For a 6-DOF arm with 50 samples/joint, the grid has 50⁶ = 15 billion cells.
- Why does RRT scale better with dimension?
- What does RRT give up compared to A* on a grid?